[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week7_breast_cancer_STARTER.ipynb)


# Week 7 -- Breast Cancer Histopathology (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** BreakHis breast cancer — binary malignant/benign, magnification domain shift, class imbalance

**Dataset:** BreakHis breast cancer histopathology

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


In [ ]:
# -- Tuesday: BreakHis Dataset Class with Slide-Level Split ----------------
# torchvision ImageFolder does not handle BreakHis's nested structure.
# BreakHis layout:
#   breakhis/malignant/slide_001/40X/image.png
#   breakhis/benign/slide_042/40X/image.png
# Critical rule: SPLIT BY SLIDE ID, not by image.
# Multiple patches from the same slide are visually nearly identical.
# Splitting by image causes data leakage and inflates AUC by 5-15%.

import os, glob
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torchvision.transforms as T

class BreakHisDataset(Dataset):
    # 0 = benign, 1 = malignant
    LABEL_MAP   = {'benign': 0, 'malignant': 1}
    CLASS_NAMES = ['Benign', 'Malignant']

    def __init__(self, root, magnification='40X', split='train',
                 transform=None, test_frac=0.2, seed=SEED):
        self.transform = transform

        # Step 1: find all image paths and extract slide_id from folder name
        records = []
        for label_name, label_idx in self.LABEL_MAP.items():
            # glob with recursive=True finds files at any depth
            pattern = os.path.join(root, label_name, '**', magnification, '*.png')
            for path in glob.glob(pattern, recursive=True):
                # path structure: .../label/slide_id/mag/image.png
                # parts[-3] is the slide folder -- our split unit
                slide_id = path.replace('\\', '/').split('/')[-3]
                records.append({'path': path, 'label': label_idx, 'slide_id': slide_id})

        if not records:  # synthetic fallback if dataset not downloaded
            print(f'No images found at {root} -- using synthetic data')
            records = [{'path': f'img_{i}.png', 'label': i%2,
                        'slide_id': f'slide_{i//5}'} for i in range(200)]

        df = pd.DataFrame(records)

        # Step 2: split by slide_id -- the critical step
        # slide_labels: one label per slide (all patches from a slide = same label)
        unique_slides = df['slide_id'].unique()
        slide_labels  = df.drop_duplicates('slide_id').set_index('slide_id')['label']

        # train_test_split on SLIDES, stratified by label to preserve class balance
        train_slides, test_slides = train_test_split(
            unique_slides, test_size=test_frac, random_state=seed,
            stratify=slide_labels[unique_slides])

        # Select only images whose slide is in the requested split
        slides = train_slides if split == 'train' else test_slides
        self.data = df[df['slide_id'].isin(slides)].reset_index(drop=True)

        print(f'BreakHis [{split}] mag={magnification}')
        print(f'  Slides: {len(slides)} | Images: {len(self.data)}')
        print(f'  Benign: {(self.data.label==0).sum()} | Malignant: {(self.data.label==1).sum()}')

    def __len__(self):
        # DataLoader calls this to know how many samples are in the dataset
        return len(self.data)

    def __getitem__(self, idx):
        # DataLoader calls this for each sample index in a batch
        row = self.data.iloc[idx]
        try:
            img = Image.open(row['path']).convert('RGB')
        except (FileNotFoundError, OSError):
            # fallback: noise image with H&E-like purple tint
            arr = np.random.randint(180, 255, (224, 224, 3), dtype='uint8')
            img = Image.fromarray(arr)
        if self.transform:
            img = self.transform(img)  # returns CHW float32 tensor
        return img, int(row['label'])  # DataLoader collects these into batches

# BreakHis domain-specific normalisation (computed in Week 2's EDA)
# These are NOT ImageNet stats -- use domain-specific values for medical images
BREAKHIS_MEAN = [0.787, 0.621, 0.742]   # per-channel mean for BreakHis 40x
BREAKHIS_STD  = [0.148, 0.197, 0.139]   # per-channel std

train_transform = T.Compose([
    T.Resize((224, 224)),        # resize patch to model input size
    T.RandomHorizontalFlip(),    # VALID for histopathology (no orientation)
    T.RandomVerticalFlip(),      # VALID for histopathology (same reason)
    T.RandomRotation(15),        # VALID: slides can be rotated up to ~15 deg
    # ColorJitter: USE CAUTION -- colour is diagnostically meaningful in H&E
    # Very small values only -- we must not jitter nuclei purple into pink
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),                # PIL -> CHW float32 [0,1]
    T.Normalize(BREAKHIS_MEAN, BREAKHIS_STD),  # domain-specific normalisation
])

# Create train and test splits
train_ds = BreakHisDataset('datasets/breakhis', split='train', transform=train_transform)
test_ds  = BreakHisDataset('datasets/breakhis', split='test',  transform=train_transform)

# Verify no leakage: train and test slides must have zero overlap
train_slides = set(train_ds.data['slide_id'])
test_slides  = set(test_ds.data['slide_id'])
overlap = train_slides.intersection(test_slides)
assert len(overlap) == 0, f'DATA LEAKAGE: {len(overlap)} shared slides!'
print('Leakage check: PASSED -- no slide appears in both train and test')


## Learning Objectives

By the end of this week, you will be able to:

- Load BreakHis and split by slide ID to prevent data leakage
- Train a binary malignant/benign classifier and achieve AUC > 0.85 at 40× magnification
- Measure the magnification domain shift: performance at 40× vs 100× vs 200× vs 400×
- Apply class-weighted loss to address the 4:1 benign-to-malignant imbalance
- Write a root cause analysis of the magnification gap



---

## MONDAY -- "BreakHis — The Clinical Stakes"


BreakHis contains 8 tumour subtypes: adenosis, fibroadenoma, phyllodes tumour, and tubular adenoma (benign); ductal carcinoma, lobular carcinoma, mucinous carcinoma, and papillary carcinoma (malignant). The binary task (benign vs malignant) is the clinical decision: does this patient need surgery? The 8-class task (subtype) informs treatment planning.


### Exercise 7.1 -- Clinical stakes mapping

Map each BreakHis tumour subtype to its clinical consequence: treatment, prognosis, surgical approach. Which two subtypes are most visually similar but clinically most different in treatment?


In [ ]:
# Exercise 7.1: Clinical stakes mapping
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: BreakHis — The Clinical Stakes (scaffold) --
# Class distribution in BreakHis
class_counts = {
    # Benign subtypes
    "adenosis":         444,
    "fibroadenoma":    1014,
    "phyllodes_tumor":  453,
    "tubular_adenoma":  149,
    # Malignant subtypes
    "ductal_carcinoma": 3451,
    "lobular_carcinoma":  170,
    "mucinous_carcinoma": 429,
    "papillary_carcinoma":560,
}
# Total benign: 2060  |  Total malignant: 4610
# Ratio: ~2.2:1 malignant:benign — class weighting required


### Monday Morning Moment

*Slack — Monday, 9:00am.*

**Dr. Kwame Asante:** Ngozi. Before you open the data: what do you expect the magnification gap to be?

**Ngozi Eze-Okafor:** Large. The cell morphology looks different at 40× vs 400×. At 40× you see tissue architecture — how cells are arranged. At 400× you see individual cell nuclei — nuclear size, shape, chromatin pattern.

**Dr. Kwame Asante:** Which features are more diagnostic for malignancy?

**Ngozi Eze-Okafor:** Nuclear features. Pleomorphism, hyperchromasia, mitotic figures. Those are at high magnification.

**Dr. Kwame Asante:** So a model trained at 40× is learning tissue architecture. A model trained at 400× is learning nuclear morphology. They are solving different sub-problems.

**Ngozi Eze-Okafor:** And a lab that uses 400× predominantly will see a model trained at 40× fail on features it was never taught to recognise.

**Dr. Kwame Asante:** Write that as the root cause hypothesis before you run the experiment.




---

## TUESDAY -- "Slide-Level Splitting — Preventing Data Leakage"


Multiple patches from the same slide will look similar to each other. If patches from the same slide appear in both training and test sets, the model memorises slide-level features and reports inflated accuracy. Always split by slide ID.


### Exercise 7.2 -- Slide-level split verification

Implement slide-level splitting. Verify: compute the Jaccard similarity between train and test slide IDs. It must be 0. Then compute image-level overlap as a sanity check. Report both.


In [ ]:
# Exercise 7.2: Slide-level split verification
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Slide-Level Splitting — Preventing Data Leakage (scaffold) --
import pandas as pd
from sklearn.model_selection import train_test_split

# Load slide metadata
df = pd.read_csv("datasets/breakhis/metadata.csv")
slide_ids = df["slide_id"].unique()

# Split slide IDs, not images
train_slides, test_slides = train_test_split(slide_ids, test_size=0.2, random_state=42,
    stratify=df.drop_duplicates("slide_id")["label"])

train_df = df[df["slide_id"].isin(train_slides)]
test_df  = df[df["slide_id"].isin(test_slides)]

print(f"Train: {len(train_slides)} slides, {len(train_df)} images")
print(f"Test:  {len(test_slides)}  slides, {len(test_df)}  images")
print("No slide appears in both sets ✅")



---

## WEDNESDAY -- "Magnification Domain Shift Experiment"


Train on 40× magnification. Test on 40×, 100×, 200×, and 400× without retraining. Report AUC at each magnification. The performance gradient is the domain shift gradient.


### Exercise 7.3 -- Magnification gap table

Train on 40×. Evaluate on all four magnifications. Report: AUC, sensitivity (recall for malignant), specificity (recall for benign) at each magnification. Which metric drops most sharply? What does that tell you about which type of error becomes more common at high magnification?


In [ ]:
# Exercise 7.3: Magnification gap table
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Magnification Domain Shift Experiment (scaffold) --
# Train on 40x only
train_40x = train_df[train_df["magnification"] == "40X"]
model = train_on(train_40x)

# Test at each magnification
for mag in ["40X", "100X", "200X", "400X"]:
    test_mag = test_df[test_df["magnification"] == mag]
    auc = evaluate(model, test_mag)
    print(f"  {mag}: AUC = {auc:.4f}")

# Expected output:
#   40X:  AUC = 0.88  (trained on this)
#   100X: AUC = 0.82  (slight drop)
#   200X: AUC = 0.76  (moderate drop)
#   400X: AUC = 0.67  (significant drop — domain shift)



---

## THURSDAY -- "Class Weighting for Imbalanced Medical Data"


BreakHis is imbalanced: more malignant than benign at the image level, but with rare subtypes in both groups. Class weighting in the loss function penalises errors on rare classes more heavily. For clinical deployment, false negatives (missed cancer) are more costly than false positives (unnecessary biopsy) — the clinical cost ratio should inform the class weights.


### Exercise 7.4 -- Clinical cost weighting

Assume false negative cost = £12,000 (delayed treatment) and false positive cost = £800 (unnecessary biopsy). Compute the clinically optimal class weight ratio. How does this differ from the sklearn "balanced" weight? Which threshold produces the minimum expected clinical cost?


In [ ]:
# Exercise 7.4: Clinical cost weighting
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Class Weighting for Imbalanced Medical Data (scaffold) --
import torch
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights from training labels
y_train = [0]*len(benign_train) + [1]*len(malignant_train)
weights = compute_class_weight("balanced", classes=[0,1], y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# Use in loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)
# This penalises benign misclassification more heavily
# (benign is the minority class in BreakHis)



---

## FRIDAY -- "The Friday Build: Magnification Gap Report"


Report: binary AUC at each magnification, the magnification gap (40× AUC minus 400× AUC), root cause analysis (why does the gap exist?), and one recommendation. The recommendation should be technically specific: not "collect more data" but "apply magnification-specific stain normalisation using a reference slide per magnification level".


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: Magnification Gap Report (scaffold) --
# Magnification gap report structure:
# 1. AUC at each magnification (table + line chart)
# 2. Gap size: {40x_AUC - 400x_AUC:.4f}
# 3. Root cause: 3 hypotheses with evidence
# 4. One specific recommendation with expected effect size
# 5. What would you need to test the recommendation?


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Ngozi Eze-Okafor:** AUC at 40×: 0.88. At 400×: 0.64. Gap: 0.24 AUC points. Root cause: the model learned tissue architecture features at 40×. These do not transfer to the nuclear morphology features visible at 400×. The two magnifications are functionally different visual domains.

**Nurse Folake Balogun:** Which magnification do our lab technicians use most?

**Yewande Adeyemi:** From the metadata — 200× and 400× at the Abuja partner site. 40× at Lagos General.

**Nurse Folake Balogun:** So a model trained on our Lagos data will fail at Abuja. Before we deploy.

**Dr. Kwame Asante:** That is the correct conclusion. Write the deployment recommendation.

**Ngozi Eze-Okafor:** Train separate models per magnification, or train a magnification-aware model that takes the magnification level as an input feature.



## YOUR WEEK 7 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I split BreakHis by slide ID — not by image — and verified no slide appears in both train and test.
- [ ] I can state the magnification gap (40× AUC minus 400× AUC) from memory and explain its root cause.
- [ ] I applied class weighting and can explain why the balanced weight differs from the clinical cost-optimal weight.
- [ ] I can name all 8 BreakHis tumour subtypes and distinguish benign from malignant.
- [ ] My magnification gap report contains a specific, testable recommendation.


## Challenge Task

> **Core path:** Train four models — one per magnification level. Compare their AUC on a common test set. Does a model trained at 400× transfer better to 200× than a model trained at 40×? Build a transfer matrix.

> **Advanced path:** Implement a magnification-conditioned model: concatenate a one-hot encoding of the magnification level to the penultimate layer. Does conditioning on magnification reduce the gap?


## Common Mistakes

**Splitting by image instead of by slide:** Patches from the same slide will appear in both train and test. The model memorises slide-level artefacts and reports inflated AUC.

**Using default threshold 0.5 for clinical deployment:** The optimal threshold depends on the clinical cost ratio of false positives to false negatives — not on 0.5. Always compute the cost-optimal threshold explicitly.

**Treating magnification as a nuisance variable:** The magnification encodes which features are visible. It is a domain variable, not noise. A model should know which magnification it is running on.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
